# <center>**Лабораторная работа №9** </center>

In [2]:
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import roc_auc_score, precision_recall_curve

### **1. Загрузите файл classification.csv.**

В нем записаны истинные классы объектов выборки (колонка true) и ответы некоторого классификатора (колонка predicted).

In [7]:
df = pd.read_csv('classification.csv')

### **2. Заполните таблицу ошибок классификации.**

Для этого подсчитайте величины TP, FP, FN и TN согласно их определениям. Например, FP — это количество объектов, имеющих класс 0, но отнесенных алгоритмом к классу 1. Ответ в данном вопросе — четыре числа через пробел.

In [36]:
y_true = df.iloc[:, 0].values 
y_pred = df.iloc[:, 1].values  

TP = ((y_true == 1) & (y_pred == 1)).sum()
TN = ((y_true == 0) & (y_pred == 0)).sum()
FP = ((y_true == 0) & (y_pred == 1)).sum()
FN = ((y_true == 1) & (y_pred == 0)).sum()

print(f"TP = {TP}")
print(f"FP = {FP}")
print(f"FN = {FN}")
print(f"TN = {TN}")

answer = f"{TP} {FP} {FN} {TN}"
print(answer)

with open('answer.txt', 'w') as f:
    f.write(answer)

TP = 43
FP = 34
FN = 59
TN = 64
43 34 59 64


### **3. Посчитайте основные метрики качества классификатора:**

Accuracy (доля верно угаданных) — sklearn.metrics.accuracy_score

Precision (точность) — sklearn.metrics.precision_score

Recall (полнота) — sklearn.metrics.recall_score

F-мера — sklearn.metrics.f1_score"

In [15]:
y_true = df.iloc[:, 0].values
y_pred = df.iloc[:, 1].values

accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

print(f"Accuracy = {accuracy:.4f}")
print(f"Precision = {precision:.4f}")
print(f"Recall = {recall:.4f}")
print(f"F1 = {f1:.4f}")

Accuracy = 0.5350
Precision = 0.5584
Recall = 0.4216
F1 = 0.4804


### **4. Имеется четыре обученных классификатора. В файле scores.csv записаны истинные классы и значения степени принадлежности положительному классу для каждого классификатора на некоторой выборке. Загрузите этот файл.**

для логистической регрессии — вероятность положительного класса (колонка score_logreg),

для SVM — отступ от разделяющей поверхности (колонка score_svm),

для метрического алгоритма — взвешенная сумма классов соседей (колонка score_knn),

для решающего дерева — доля положительных объектов в листе (колонка score_tree).

In [17]:
scores_df = pd.read_csv('scores.csv')
y_true = scores_df['true'].values

### **5. Посчитайте площадь под ROC-кривой для каждого классификатора.**

Какой классификатор имеет наибольшее значение метрики AUC-ROC (укажите название столбца с ответами этого классификатора)? Воспользуйтесь функцией sklearn.metrics.roc_auc_score.

In [35]:
scores_df = pd.read_csv('scores.csv')
y_true = scores_df['true'].values

classifiers = ['score_logreg', 'score_svm', 'score_knn', 'score_tree']
auc_scores = {}

for clf in classifiers:
    auc_scores[clf] = roc_auc_score(y_true, scores_df[clf].values)
    print(f"{clf}: {auc_scores[clf]:.4f}")

best = max(auc_scores, key=auc_scores.get)
print(f"Наибольшее значение метрики AUC-ROC имеет: {best}")

score_logreg: 0.7192
score_svm: 0.7087
score_knn: 0.6352
score_tree: 0.6919
Наибольшее значение метрики AUC-ROC имеет: score_logreg


### **6. Какой классификатор достигает наибольшей точности (Precision) при полноте (Recall) не менее 70% (укажите название столбца с ответами этого классификатора)?**

Какое значение точности при этом получается?

In [33]:
classifier_columns = [col for col in scores_df.columns if col != 'true']

results = {}

print(f"{'Классификатор':<20} {'Max Precision':<15} {'При Recall':<12} {'Порог':<12}")

for col in classifier_columns:
    scores = scores_df[col].values
    
    
    precision, recall, thresholds = precision_recall_curve(y_true, scores)
    
    mask = recall >= 0.7
    if mask.any():
        
        max_precision = precision[mask].max()
        
        idx = np.argmax(precision[mask])
        corresponding_recall = recall[mask][idx]
        
        if idx < len(thresholds):
            threshold = thresholds[idx]
        else:
            threshold = thresholds[-1] if len(thresholds) > 0 else None
        
        results[col] = {
            'max_precision': max_precision,
            'recall': corresponding_recall,
            'threshold': threshold
        }
        
        print(f"{col:<20} {max_precision:.4f}         {corresponding_recall:.4f}     {threshold:.4f}")
    else:
        results[col] = {
            'max_precision': 0.0,
            'recall': 0.0,
            'threshold': None
        }
        print(f"{col:<20} {'Нет точек':<15} {'---':<12} {'---':<12}")


best_classifier = max(results, key=lambda x: results[x]['max_precision'])
best_precision = results[best_classifier]['max_precision']
best_recall = results[best_classifier]['recall']

print(f"\n Лучший классификатор:")
print(f"   {best_classifier}")
print(f"   Значение точности = {best_precision:.4f}")

Классификатор        Max Precision   При Recall   Порог       
score_logreg         0.6303         0.7653     0.4885
score_svm            0.6228         0.7245     -0.1235
score_knn            0.6066         0.7551     0.3193
score_tree           0.6518         0.7449     0.4000

 Лучший классификатор:
   score_tree
   Значение точности = 0.6518
